# tensor-zeros-init — ex5: allocate output buffer, then paint hits

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-zeros-init`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five allocation patterns that ramp from `torch.zeros(n)` → multi-axis shape → `zeros_like` → dtype-long index buffer → allocate-then-scatter for the canonical Ray Tracing per-ray output-buffer pattern. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `tensor-zeros-init`**, which bridges to the bank subtopic `Numpy: Core array literacy` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-zeros-init"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Tensor allocation — quick refresher

**The four shapes of `zeros`:**
- `t.zeros(n)` — 1-D, shape `(n,)`, default `float32`.
- `t.zeros(b, h, w)` — multi-axis positional args.
- `t.zeros_like(x)` — mirror `x.shape` + `x.dtype` + `x.device`.
- `t.zeros(n, dtype=t.long)` — override dtype for index buffers.

**The accumulator pattern.** Allocate the right-shaped zero buffer first; scatter per-element results into it via indexed assignment. Cleaner and faster than `append`-and-stack.

### Exercise 5 — allocate output buffer, then paint hits

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize shape arg + dtype default + indexed assignment to scatter per-ray hit colors into an output buffer.
> Keywords: accumulator, indexed-assign, ray-tracing, multi-kc
> ```

**KCs targeted:** `zeros-multi-axis-shape`, `zeros-dtype-control`, `zeros-allocate-then-fill`

Implement `ex5_paint_hits(num_rays, hit_indices, hit_colors)`. The canonical Ray Tracing output-buffer pattern:

1. Allocate a `(num_rays, 3)` zero buffer (float32 by default — perfect for RGB colors in `[0, 1]`).
2. For each `k`, write `hit_colors[k]` into row `hit_indices[k]`.
3. Rays not in `hit_indices` stay `[0, 0, 0]` (black — no hit).

Inputs:
- `num_rays`: int.
- `hit_indices`: 1-D long tensor, shape `(K,)`, values in `[0, num_rays)`.
- `hit_colors`: 2-D float tensor, shape `(K, 3)`.

Output: `(num_rays, 3)` float32 tensor.

Hint: `out[hit_indices] = hit_colors` does the scatter in one shot.

> ⚠️ **Integrative exercise.** This combines 3 KCs (shape allocation, default dtype, indexed assignment); empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step up vs Exercises 1-4.

In [ ]:
def ex5_paint_hits(num_rays: int, hit_indices: Tensor, hit_colors: Tensor) -> Tensor:
    """Allocate (num_rays, 3) zero buffer; write hit_colors at hit_indices."""
    raise NotImplementedError()


def _test_ex5():
    hit_indices = t.tensor([0, 2, 3], dtype=t.long)
    hit_colors = t.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
    out = ex5_paint_hits(5, hit_indices, hit_colors)
    assert out.shape == (5, 3), f'expected (5, 3), got {tuple(out.shape)}'
    assert out.dtype == t.float32, f'expected float32, got {out.dtype}'
    expected = t.tensor([
        [1.0, 0.0, 0.0],  # ray 0 — red hit
        [0.0, 0.0, 0.0],  # ray 1 — no hit, stays zero
        [0.0, 1.0, 0.0],  # ray 2 — green hit
        [0.0, 0.0, 1.0],  # ray 3 — blue hit
        [0.0, 0.0, 0.0],  # ray 4 — no hit, stays zero
    ])
    assert t.allclose(out, expected), f'value mismatch:\n{out}\nvs\n{expected}'
    # Edge case — no hits at all.
    empty_idx = t.zeros(0, dtype=t.long)
    empty_col = t.zeros(0, 3)
    out_empty = ex5_paint_hits(3, empty_idx, empty_col)
    assert out_empty.shape == (3, 3) and t.all(out_empty == 0), 'no-hits case must return all-zero buffer'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_paint_hits(num_rays: int, hit_indices: Tensor, hit_colors: Tensor) -> Tensor:
    out = t.zeros(num_rays, 3)
    out[hit_indices] = hit_colors
    return out
```

**Why this pattern matters.** Every per-ray Ray Tracing computation uses this shape: allocate a `(num_rays, ...)` output buffer with the right dtype, compute the mask of which rays did something, scatter the per-hit values back in. Rays that don't hit anything keep the default fill (zero / -inf / NaN sentinel depending on the use).

**Why the indexed-assign works.** `out[hit_indices] = hit_colors` uses advanced indexing: PyTorch evaluates `hit_indices` as a list of row positions and writes the matching row from `hit_colors` into each. Requires `hit_indices.dtype == long` — see Exercise 4.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()